# Notebook Colab (T4) — RAG Formulaire avec Meta Llama 3 8B

Ce notebook utilise **Meta Llama 3 8B Instruct** au lieu de Mistral 7B pour une meilleure compréhension des instructions en français.

## Avantages de Llama 3 8B

- ✅ **Meilleure instruction-following** : Plus fidèle aux prompts
- ✅ **Contexte plus large** : 8192 tokens vs 4096 pour Mistral
- ✅ **Multilingue amélioré** : Meilleur support du français
- ✅ **Raisonnement** : Meilleures capacités de réflexion

Ce notebook permet de :
- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index.
- Poser des questions avec Llama 3 8B au lieu de Mistral 7B.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 30) pour accélérer l'ingestion sur Colab.

> **Note :** Ce notebook utilise le code intégré directement depuis le dépôt avec toutes les optimisations récentes.

## 1) Vérifier le GPU

In [ ]:
!nvidia-smi

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.

In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.

In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q \
    requests beautifulsoup4 tqdm langdetect pydantic rank-bm25 \
    chromadb sentence-transformers scikit-learn transformers torch \
    click docling pdfplumber \
    pypdf pymupdf \
    bitsandbytes accelerate
!pip install -q pikepdf


## 4) Paramétrage avec Llama 3 8B

**Configuration spécifique pour Llama 3 8B:**
- Modèle plus grand mais meilleur instruction-following
- Contexte 8K tokens (vs 4K pour Mistral)
- Quantification 4-bit pour tenir sur T4 (15GB VRAM)

Variables d'environnement ajustées:
- `RAG_FORM_GEN_MODEL`: Meta Llama 3 8B Instruct
- `RAG_FORM_MIN_FORMS`: 30 formulaires pour accélérer
- `RAG_FORM_GEN_4BIT`: Activer quantification 4-bit

In [ ]:
# ============================================================================\
# 🔑 HUGGING FACE AUTHENTICATION
# ============================================================================\
# This is required to download the gated Mistral model

from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    import getpass
    HF_TOKEN = getpass.getpass("Entrez votre token Hugging Face (hf_...): ")

login(token=HF_TOKEN)

print("✅ Hugging Face login successful.")

In [ ]:
from pprint import pprint
import os

# Ensure token is in env for libraries that check it directly
if "HF_TOKEN" not in os.environ and "HF_TOKEN" in locals():
    os.environ["HF_TOKEN"] = HF_TOKEN

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

# 🦙 Configuration pour Meta Llama 3 8B Instruct
#os.environ["RAG_FORM_GEN_MODEL"] = "meta-llama/Meta-Llama-3-8B-Instruct"
os.environ["RAG_FORM_GEN_MODEL"] = "meta-llama/Llama-3.1-8B-Instruct"

os.environ["RAG_FORM_GEN_4BIT"] = "true"  # Obligatoire pour T4

# Optimisations récentes
os.environ.setdefault("RAG_FORM_CHUNK_SIZE", "400")  # Chunks plus grands
os.environ.setdefault("RAG_FORM_CHUNK_OVERLAP", "80")  # Meilleur contexte
os.environ.setdefault("RAG_FORM_STRICT_VERIFICATION", "false")  # Mode lenient + form code validation

print("Configuration Llama 3 8B en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})

## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires, découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.

> **Optimisations intégrées:**
> - LLM Singleton: Une seule instance du modèle (économise ~50% de mémoire)
> - Chunks 400 tokens: Meilleur contexte que 200 tokens
> - Downloader Deduplication: Évite les doublons dans le manifest
> - Smart Retrieval: Détection automatique des codes de formulaire spécifiques
> - Form Code Validation: Empêche les hallucinations de codes formulaire

In [ ]:
from rag_formulaire.ingest import complete_reindex

# Exécuter la ré-indexation complète (Nettoyage -> Téléchargement -> Indexation -> Validation)
complete_reindex()

### Aperçu du manifest

In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)

## 6) Charger le LLM Llama 3 8B

Le Llama 3 8B sera chargé avec le singleton pattern - une seule instance partagée.

**Différences vs Mistral 7B:**
- Plus grande fenêtre de contexte (8K vs 4K)
- Meilleur instruction-following en français
- Légèrement plus lent mais plus précis

In [ ]:
from rag_formulaire.pipeline import RAGPipeline
import os

# HF_TOKEN should be set in step 4

print("🦙 Chargement du pipeline RAG avec Llama 3...")
pipeline = RAGPipeline()
print("✅ Pipeline chargé !")

### Utilitaires d'affichage

Fonction pour afficher les résultats de manière formatée avec Markdown.

In [ ]:
from rag_formulaire.notebook_utils import display_result

### Gestion de la mémoire GPU

Outils pour nettoyer le cache GPU entre les requêtes et éviter les erreurs OOM (Out Of Memory).

> **Note:** Llama 3 8B utilise plus de VRAM que Mistral 7B. Le nettoyage est encore plus important.

In [ ]:
from rag_formulaire.notebook_utils import force_cleanup

force_cleanup()

## 7) Tests avec questions variées

Testez le pipeline avec une série de questions pour comparer Llama 3 8B vs Mistral 7B.

In [ ]:
from rag_formulaire.notebook_utils import force_cleanup, display_result
import os
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()  # Bloque les warnings "generation flags"

# On garde un contexte court pour la mémoire
os.environ["RAG_FORM_FINAL_EVIDENCE_K"] = "3"

test_questions = [
    # --- ✅ PARTIE 1 : Questions sur des documents INDEXÉS (Doivent réussir) ---
    "Qui doit signer le formulaire IMM 5476 pour désigner un représentant ?",
    "Que doit-on déclarer à propos des maladies mentales dans le questionnaire médical IMM 5955 ?",
    "Quels documents peuvent servir de preuve d'expérience de travail au Canada selon le formulaire IMM 0134 ?",
    "À qui s'adresse l'offre d'emploi pour les ressortissants étrangers dispensés d'EIMT (IMM 0116) ?",
    "Quel est le rôle de l'interprète décrit dans le formulaire IMM 5744 ?",
    "Quelles informations l'employeur doit-il fournir sur l'adresse commerciale dans le formulaire IMM 0267 ?",
    "Quelles sont les responsabilités de l'employeur concernant l'offre d'emploi dans le formulaire IMM 0273 ?",
    "Dans quel cas les frais relatifs au droit de résidence permanente sont-ils remboursés (IMM 5741) ?",

    # --- ❌ PARTIE 2 : Questions "Test de Sécurité" (Documents ABSENTS) ---
    "Quels sont les documents requis dans la liste de contrôle IMM 5488 ?",
    "Qui doit être listé dans le formulaire de renseignements sur la famille IMM 5707 ?"
]

print(f"🚀 Lancement du test hybride ({len(test_questions)} questions)...\n")
force_cleanup()

for i, question in enumerate(test_questions, 1):
    print(f"▶️ Question {i}/{len(test_questions)}: {question}")
    try:
        # Appel du pipeline
        result = pipeline.ask_question(question)
        
        # Affichage simplifié pour le log console
        answer = result.get('answer', '')
        evidence = result.get('evidence', [])
        if evidence:
            sources = list(set([c.base_chunk.form_code for c in evidence]))
            print(f"   ✅ Sources : {sources}")
        else:
            print("   ⚠️ Aucun extrait trouvé.")
            
        # Affichage riche (Markdown)
        display_result(result)
        
    except Exception as e:
        print(f"   ⚠️ Erreur : {e}")
    
    print("-" * 50)
    force_cleanup()